[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/bases-de-datos-vectoriales/01-pgvector.ipynb)

# Búsqueda Vectorial con pgvector y PostgreSQL

> Aprende a almacenar y buscar embeddings directamente en PostgreSQL usando pgvector,
> e integra un sistema RAG completo sobre tu base de datos existente.

## Objetivos
- Instalar y configurar la extensión pgvector en PostgreSQL
- Crear tablas con columnas de tipo vector
- Indexar documentos con embeddings de sentence-transformers
- Construir un sistema RAG completo con Claude y pgvector

**Requisitos:** PostgreSQL >= 14 instalado con la extensión pgvector.

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic psycopg2-binary pgvector sentence-transformers --quiet

## 2. Conexión a PostgreSQL y activar pgvector

Ajusta `DB_CONFIG` con tus credenciales locales. Si no tienes pgvector, el notebook
usa ChromaDB como fallback para que puedas seguir el flujo igualmente.

In [ ]:
import anthropic
from sentence_transformers import SentenceTransformer
import numpy as np

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print("Cargando modelo de embeddings...")
modelo_emb = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
DIM = 384  # Dimensión del modelo MiniLM

# Configuración de PostgreSQL
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "vectordb",
    "user": "postgres",
    "password": "postgres"
}

# Intentar conectar a PostgreSQL
conn = None
try:
    import psycopg2
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    conn.commit()
    print(f"✓ Conectado a PostgreSQL. Extensión pgvector activa.")
except Exception as e:
    print(f"PostgreSQL no disponible ({e}). Usando ChromaDB como alternativa.")
    import chromadb
    chroma = chromadb.EphemeralClient()
    coleccion = chroma.create_collection("documentos")
    print("✓ ChromaDB inicializado como alternativa.")

## 3. Crear tabla con columna vector e indexar documentos

In [ ]:
# Corpus de documentos sobre IA
DOCUMENTOS = [
    {"titulo": "Introducción al ML", "contenido": "El machine learning permite a los sistemas aprender de datos sin programación explícita."},
    {"titulo": "Redes Neuronales", "contenido": "Las redes neuronales imitan el funcionamiento del cerebro con capas de neuronas artificiales."},
    {"titulo": "Transformers", "contenido": "Los transformers usan atención multi-cabezal para procesar secuencias en paralelo."},
    {"titulo": "RAG", "contenido": "RAG combina recuperación de información con generación de texto para respuestas fundamentadas."},
    {"titulo": "Embeddings", "contenido": "Los embeddings son vectores densos que capturan el significado semántico del texto."},
    {"titulo": "Fine-tuning", "contenido": "El fine-tuning adapta modelos preentrenados a tareas específicas con menos datos."},
    {"titulo": "Prompt Engineering", "contenido": "El prompt engineering diseña instrucciones efectivas para guiar el comportamiento de los LLMs."},
    {"titulo": "Agentes IA", "contenido": "Los agentes de IA combinan LLMs con herramientas externas para ejecutar tareas complejas."},
    {"titulo": "Visión por Computador", "contenido": "Los modelos multimodales procesan imágenes y texto de forma conjunta."},
    {"titulo": "LoRA", "contenido": "LoRA reduce los parámetros entrenables durante el fine-tuning usando matrices de bajo rango."},
]

# Generar embeddings
textos = [d["contenido"] for d in DOCUMENTOS]
embeddings = modelo_emb.encode(textos)

if conn:  # PostgreSQL disponible
    cur = conn.cursor()
    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS documentos (
            id SERIAL PRIMARY KEY,
            titulo TEXT,
            contenido TEXT,
            embedding vector({DIM})
        );
    """)
    cur.execute("DELETE FROM documentos;")  # Limpiar tabla

    for doc, emb in zip(DOCUMENTOS, embeddings):
        cur.execute(
            "INSERT INTO documentos (titulo, contenido, embedding) VALUES (%s, %s, %s)",
            (doc["titulo"], doc["contenido"], emb.tolist())
        )

    # Crear índice HNSW para búsqueda aproximada rápida
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_documentos_hnsw
        ON documentos USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 64);
    """)
    conn.commit()
    print(f"✓ {len(DOCUMENTOS)} documentos indexados en PostgreSQL con índice HNSW.")

else:  # Fallback ChromaDB
    coleccion.add(
        ids=[str(i) for i in range(len(DOCUMENTOS))],
        documents=textos,
        embeddings=embeddings.tolist(),
        metadatas=[{"titulo": d["titulo"]} for d in DOCUMENTOS]
    )
    print(f"✓ {len(DOCUMENTOS)} documentos indexados en ChromaDB.")

## 4. Búsqueda por similitud coseno

In [ ]:
def buscar_similares(query: str, n: int = 3) -> list[dict]:
    """Busca documentos similares por similitud coseno."""
    query_emb = modelo_emb.encode([query])[0]

    if conn:  # pgvector: operador <=> = distancia coseno
        cur = conn.cursor()
        cur.execute("""
            SELECT titulo, contenido, 1 - (embedding <=> %s::vector) AS similitud
            FROM documentos
            ORDER BY embedding <=> %s::vector
            LIMIT %s;
        """, (query_emb.tolist(), query_emb.tolist(), n))
        return [{"titulo": r[0], "contenido": r[1], "similitud": float(r[2])} for r in cur.fetchall()]
    else:  # ChromaDB
        res = coleccion.query(query_embeddings=[query_emb.tolist()], n_results=n)
        return [{"titulo": m["titulo"], "contenido": d, "similitud": 1 - dist}
                for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

QUERIES = [
    "¿Cómo aprenden las máquinas a partir de datos?",
    "¿Qué técnica reduce los parámetros de entrenamiento?",
    "¿Cómo se representan las palabras como vectores?"
]

for query in QUERIES:
    print(f"\nQuery: '{query}'")
    resultados = buscar_similares(query, n=3)
    for i, r in enumerate(resultados, 1):
        print(f"  {i}. [{r['similitud']:.3f}] {r['titulo']}: {r['contenido'][:70]}...")

## 5. RAG completo: recuperar contexto → Claude genera respuesta

In [ ]:
def rag_responder(pregunta: str, n_contextos: int = 3) -> str:
    """RAG: recupera documentos relevantes de pgvector y genera respuesta con Claude."""
    contextos = buscar_similares(pregunta, n_contextos)
    contexto_str = "\n".join([f"[{c['titulo']}] {c['contenido']}" for c in contextos])

    prompt = f"""Responde usando ÚNICAMENTE la información del contexto. Si no está en el contexto, dilo.

Contexto recuperado:
{contexto_str}

Pregunta: {pregunta}"""

    return client.messages.create(
        model=MODELO, max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text

preguntas_rag = [
    "¿Qué es RAG y cómo funciona?",
    "¿Cuál es la diferencia entre embeddings y fine-tuning?",
    "¿Cuál es la capital de Francia?"  # Fuera del contexto
]

print("=== SISTEMA RAG CON PGVECTOR ===")
for pregunta in preguntas_rag:
    print(f"\nPregunta: {pregunta}")
    respuesta = rag_responder(pregunta)
    print(f"Respuesta: {respuesta[:300]}")

if conn:
    conn.close()